# Elastic P and S waves

*fluids · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/elastic_waves)

The Navier–Cauchy equation for an elastic solid. A short push at one point sends out two waves: a fast compressional P wave (seen in $\nabla\cdot\mathbf u$) and a slower shear S wave (seen in $\nabla\times\mathbf u$). Both terms are contractions of one table of second derivatives $\partial_i\partial_j u_k$ — two einsums with different repeated indices.

**Vector form**

$$\rho\,\ddot{\mathbf u} = (\lambda + \mu)\,\nabla(\nabla\cdot\mathbf u) + \mu\,\nabla^2\mathbf u + \mathbf f$$

**Index (tensor) form**

$$\rho\,\ddot u_i = (\lambda + \mu)\,\partial_i\partial_j u_j + \mu\,\partial_j\partial_j u_i + f_i,\qquad c_P = \sqrt{\frac{\lambda + 2\mu}{\rho}},\quad c_S = \sqrt{\frac{\mu}{\rho}},\qquad \Delta t\,\sqrt{c_P^2 + c_S^2} \le \Delta x$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_i$ | `u` | (2, N, N) | displacement field, component $i \in \{x, y\}$ at every node |
| $H_{ijk} = \partial_i\partial_j u_k$ | `hessian(u, h)` | (2, 2, 2, N, N) | all second derivatives by central differences |
| $\partial_i\partial_j u_j$ | `einsum('ijj...->i...', H)` | (2, N, N) | repeated $j$ (2nd and 3rd slot) ⇒ gradient of the divergence |
| $\partial_j\partial_j u_i$ | `einsum('jji...->i...', H)` | (2, N, N) | repeated $j$ (1st and 2nd slot) ⇒ Laplacian of each component |
| $f_i$ | `ricker(t, f0) * point_force(...)` | (2, N, N) | a short Mexican-hat push at the centre along one direction |
| $\partial_j u_j,\ \epsilon_{ij}\partial_i u_j$ | `divergence(u, h), curl(u, h)` | (N, N) | volume change (P) and rotation (S) |
| $\Delta t$ | `stable_time_step(h, c_p, c_s, S)` | scalar | leapfrog step; a Courant number $S > 1$ is refused because the scheme blows up |

Units with $\rho = 1$ and cell size 1. Time stepping is leapfrog, $u^{n+1} = 2u^n - u^{n-1} + \Delta t^2\,\ddot u^n$. Taking the divergence of the equation gives a wave equation for $\theta = \partial_j u_j$ with speed $c_P$; taking the curl gives one for the rotation with speed $c_S < c_P$ — in rock $c_P \approx 1.7\,c_S$, which is why an earthquake's P wave arrives first and the S wave seconds later. A point force radiates P waves mostly along the force and S waves mostly across it, as the two lobed patterns show. The dashed circles are the predicted fronts $c\,(t - t_0)$, where $t_0$ is the peak of the source wavelet. A damping border absorbs waves before they reach the edge (real codes use a perfectly matched layer); a Poisson ratio near ½ ($\lambda \gg \mu$) makes $c_P$ large and forces a small time step.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/elastic_waves/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import stable_time_step, wave_speeds

N, H, BORDER, RHO = 181, 1.0, 20, 1.0  # grid cells per side, cell size, absorbing border width, density
F0 = 0.08  # peak frequency of the source wavelet


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lam, mu = params["lam"], params["mu"]
    c_p, c_s = wave_speeds(lam, mu, RHO)
    dt = stable_time_step(H, c_p, c_s, params["courant"])
    u, u_prev = tensors.at_rest(N, dev)  # (2, N, N) each
    force = tensors.point_force(N, params["direction"], 1.5, dev)  # (2, N, N)
    damping = tensors.sponge(N, BORDER, 0.08, dev)  # (N, N)

    result = solve.simulate(u, u_prev, force, damping, H, dt, lam, mu, RHO, F0, params["t_max"])

    t0 = 1.2 / F0  # the wavelet peaks at t0
    times = result["times"]
    print(f"c_P = sqrt((λ+2μ)/ρ) = {c_p:.3f},  c_S = sqrt(μ/ρ) = {c_s:.3f},  c_P/c_S = {c_p / c_s:.3f}")
    print(f"dt = {dt:.3f} (Courant number {params['courant']}),  {len(times) - 1} leapfrog steps")
    if params["direction"] == 0.0:
        a, b = int(len(times) * 0.6), len(times) - 1
        dtt = (times[b] - times[a]).item()
        cp_m = (solve.front_radius(result["div"][b], "x") - solve.front_radius(result["div"][a], "x")) * H / dtt
        cs_m = (solve.front_radius(result["curl"][b], "y") - solve.front_radius(result["curl"][a], "y")) * H / dtt
        print(f"measured front speeds: P {cp_m:.2f} (along the force), S {cs_m:.2f} (across it)")
    return plot.animate(result, c_p, c_s, t0, BORDER, params["direction"], out_dir / "elastic_waves.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/elastic_waves/equation.py
r"""Physics → tensor translation.

The Navier–Cauchy equation: Newton's law for a homogeneous isotropic elastic solid,

    ρ \ddot u_i = (λ + μ) ∂_i ∂_j u_j + μ ∂_j ∂_j u_i + f_i        (j summed)

On the grid we build the table of second derivatives H_{ijk} = ∂_i ∂_j u_k at every node
(central differences), then both terms are contractions of H:

    ∂_i ∂_j u_j = einsum('ijj...->i...', H)      (gradient of the divergence → P waves)
    ∂_j ∂_j u_i = einsum('jji...->i...', H)      (Laplacian of each component → S waves)

Taking the divergence or curl of the equation splits it into two wave equations with speeds

    c_P = \sqrt{(λ + 2μ)/ρ}  for θ = ∂_j u_j,        c_S = \sqrt{μ/ρ}  for ω = ε_{ij} ∂_i u_j.
"""

import math

import torch


def d1(f: torch.Tensor, axis: int, h: float) -> torch.Tensor:
    """Central first derivative along grid axis 0 (x) or 1 (y) of a field (..., Nx, Ny)."""
    dim = f.dim() - 2 + axis
    return (torch.roll(f, -1, dim) - torch.roll(f, 1, dim)) / (2 * h)


def hessian(u: torch.Tensor, h: float) -> torch.Tensor:
    """H_{ijk} = ∂_i ∂_j u_k by central differences.   u: (2, Nx, Ny) → (2, 2, 2, Nx, Ny)."""
    dxx = (torch.roll(u, -1, 1) - 2 * u + torch.roll(u, 1, 1)) / h**2
    dyy = (torch.roll(u, -1, 2) - 2 * u + torch.roll(u, 1, 2)) / h**2
    dxy = d1(d1(u, 0, h), 1, h)
    return torch.stack([torch.stack([dxx, dxy]), torch.stack([dxy, dyy])])  # [i, j, k, x, y]


def acceleration(u: torch.Tensor, h: float, lam: float, mu: float, rho: float) -> torch.Tensor:
    r"""\ddot u_i = [(λ+μ) ∂_i∂_j u_j + μ ∂_j∂_j u_i] / ρ.   (2, Nx, Ny) → (2, Nx, Ny)"""
    H = hessian(u, h)
    grad_div = torch.einsum("ijj...->i...", H)  # ∂_i ∂_j u_j
    laplace = torch.einsum("jji...->i...", H)  # ∂_j ∂_j u_i
    return ((lam + mu) * grad_div + mu * laplace) / rho


def divergence(u: torch.Tensor, h: float) -> torch.Tensor:
    """θ = ∂_j u_j (volume change, carried by P waves): (2, Nx, Ny) → (Nx, Ny)."""
    return d1(u[0], 0, h) + d1(u[1], 1, h)


def curl(u: torch.Tensor, h: float) -> torch.Tensor:
    """ω = ∂_x u_y - ∂_y u_x (rotation, carried by S waves): (2, Nx, Ny) → (Nx, Ny)."""
    return d1(u[1], 0, h) - d1(u[0], 1, h)


def wave_speeds(lam: float, mu: float, rho: float) -> tuple[float, float]:
    """c_P = sqrt((λ + 2μ)/ρ), c_S = sqrt(μ/ρ)."""
    return math.sqrt((lam + 2 * mu) / rho), math.sqrt(mu / rho)


def stable_time_step(h: float, c_p: float, c_s: float, courant: float) -> float:
    """Δt = S h / sqrt(c_P² + c_S²). The explicit scheme is only stable for S ≤ 1, so larger S is refused."""
    if courant > 1.0:
        raise ValueError(f"Courant number {courant} > 1: the leapfrog scheme would blow up")
    return courant * h / math.sqrt(c_p**2 + c_s**2)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/elastic_waves/tensors.py
"""Build the concrete tensors: the displacement field, the point-force source and the absorbing border."""

import math

import torch


def at_rest(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Displacement u_i at the current and previous time step, both zero: (2, N, N) each."""
    return torch.zeros(2, n, n, device=device), torch.zeros(2, n, n, device=device)


def point_force(n: int, direction_deg: float, width: float, device: torch.device) -> torch.Tensor:
    """Spatial pattern of the source: a small Gaussian blob at the centre pushing along one direction.

    Returns f_i(x, y) of shape (2, N, N) with unit peak strength.
    """
    c = (n - 1) / 2
    x = torch.arange(n, device=device, dtype=torch.float32) - c
    blob = torch.exp(-(x[:, None] ** 2 + x[None, :] ** 2) / (2 * width**2))  # (N, N)
    a = math.radians(direction_deg)
    return torch.stack([math.cos(a) * blob, math.sin(a) * blob])


def ricker(t: float, f0: float) -> float:
    """Ricker (Mexican-hat) wavelet in time with peak frequency f0, centred at t0 = 1.2 / f0."""
    s = (math.pi * f0 * (t - 1.2 / f0)) ** 2
    return (1 - 2 * s) * math.exp(-s)


def sponge(n: int, width: int, strength: float, device: torch.device) -> torch.Tensor:
    """Damping factor (N, N): 1 in the interior, smoothly below 1 near the edges so outgoing waves fade away."""
    i = torch.arange(n, device=device, dtype=torch.float32)
    depth = torch.clamp(width - torch.minimum(i, n - 1 - i), min=0) / width  # 0 inside, → 1 at the edge
    d1 = 1 - strength * depth**2
    return d1[:, None] * d1[None, :]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/elastic_waves/solve.py
"""Leapfrog time stepping of the Navier–Cauchy equation, saving div u and curl u."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import acceleration, curl, divergence


def simulate(
    u: torch.Tensor, u_prev: torch.Tensor, force: torch.Tensor, damping: torch.Tensor, h: float, dt: float,
    lam: float, mu: float, rho: float, f0: float, t_max: float,
) -> dict:  # fmt: skip
    """u^{n+1} = 2u^n - u^{n-1} + Δt² (\\ddot u + f/ρ), damped near the border.

    Returns times (T,), div u (T, N, N) and curl u (T, N, N).
    """
    n_steps = int(t_max / dt)
    times, divs, curls = [0.0], [divergence(u, h)], [curl(u, h)]
    for n in steps(n_steps, "time steps"):
        source = tensors.ricker(n * dt, f0) * force / rho
        u_next = 2 * u - u_prev + dt**2 * (acceleration(u, h, lam, mu, rho) + source)
        u_prev, u = u * damping, u_next * damping
        times.append((n + 1) * dt)
        divs.append(divergence(u, h))
        curls.append(curl(u, h))
    return {"times": torch.tensor(times), "div": torch.stack(divs), "curl": torch.stack(curls), "u": u}


def front_radius(field: torch.Tensor, direction: str) -> float:
    """Distance from the centre to the strongest |field| along the +x axis or the +y axis (in cells)."""
    n = field.shape[-1]
    c = n // 2
    line = field[c:, c] if direction == "x" else field[c, c:]
    return float(line.abs().argmax())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/elastic_waves/plot.py
"""Side by side: |div u| (P waves) and |curl u| (S waves), with circles at the predicted fronts c (t - t0)."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, Rectangle

from problems._common.plotting import frame_indices, save_gif


def animate(result: dict, c_p: float, c_s: float, t0: float, border: int, direction_deg: float, path: Path) -> Path:
    t = result["times"].cpu().numpy()
    fields = [result["div"].abs().cpu().numpy(), result["curl"].abs().cpu().numpy()]  # (T, N, N) each
    n = fields[0].shape[-1]
    centre = (n - 1) / 2
    arrow = 12 * np.array([math.cos(math.radians(direction_deg)), math.sin(math.radians(direction_deg))])

    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.9))
    images, circles = [], []
    for ax, field, speed, name, cmap in zip(axes, fields, (c_p, c_s), ("|div u|  (P wave)", "|curl u|  (S wave)"),
                                            ("Blues", "Reds")):  # fmt: skip
        top = np.quantile(field, 0.999) + 1e-12
        images.append(ax.imshow(field[0].T, origin="lower", cmap=plt.get_cmap(cmap, 10), vmin=0, vmax=top,
                                extent=(0, n, 0, n), interpolation="nearest"))  # fmt: skip
        circles.append(ax.add_patch(Circle((centre, centre), 0, fill=False, ec="#111827", ls="--", lw=1.0)))
        ax.add_patch(
            Rectangle((border, border), n - 2 * border, n - 2 * border, fill=False, ec="#9ca3af", ls=":", lw=0.8)
        )
        ax.annotate("", (centre + arrow[0], centre + arrow[1]), (centre - arrow[0], centre - arrow[1]),
                    arrowprops={"arrowstyle": "->", "color": "#16a34a", "lw": 1.5})  # fmt: skip
        ax.set_title(f"{name}   dashed: c t′, c = {speed:.2f}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    clock = fig.suptitle(" ", fontsize=10)

    def update(step: int):
        for image, circle, field, speed in zip(images, circles, fields, (c_p, c_s)):
            image.set_data(field[step].T)
            circle.set_radius(max(0.0, speed * (t[step] - t0)))  # t′ = t - t0: time since the wavelet peak
        clock.set_text(f"point force (green arrow) in an elastic solid   t = {t[step]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'lam': 2.0,
    'mu': 1.0,
    'direction': 0.0,
    'courant': 0.6,
    't_max': 45.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.elastic_waves")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.elastic_waves.main")
problem_dir = Path("problems/fluids/elastic_waves")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
